In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [324]:

train = pd.read_csv("../raw_data/train.csv")
test = pd.read_csv("../raw_data/test.csv")

In [325]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [326]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [327]:
columns = [col for col in train.columns if col not in ("Survived", "PassengerId")]
X = train[columns]
y = train["Survived"]
X_test = test[columns]
test_ids = test["PassengerId"]

In [328]:
X.head()

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [329]:
# Видим пропуски в столбце Age
X.describe()

,Pclass,Age,SibSp,Parch,Fare
count,891.000000,714.000000,891.000000,891.000000,891.000000
mean,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.420000,0.000000,0.000000,0.000000
25%,2.000000,20.125000,0.000000,0.000000,7.910400
50%,3.000000,28.000000,0.000000,0.000000,14.454200
75%,3.000000,38.000000,1.000000,0.000000,31.000000
max,3.000000,80.000000,8.000000,6.000000,512.329200


In [330]:
X.describe(include="object")

/var/folders/6n/xcr_g_3d13529x7vjx2_z7qc0000gn/T/ipykernel_74205/3236359428.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.describe(include="object")


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


In [331]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Name      891 non-null    str    
 2   Sex       891 non-null    str    
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Ticket    891 non-null    str    
 7   Fare      891 non-null    float64
 8   Cabin     204 non-null    str    
 9   Embarked  889 non-null    str    
dtypes: float64(2), int64(3), str(5)
memory usage: 69.7 KB


In [332]:
X["Sex"].value_counts()

Sex
male      577
female    314
Name: count, dtype: int64

In [333]:
X["Sex"] = X["Sex"].map({"male": 1, "female": 0})
X_test["Sex"] = X_test["Sex"].map({"male": 1, "female": 0})

In [334]:
X["Sex"].sum()

np.int64(577)

In [335]:
X.head()

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,C85,C
2,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1,0,113803,53.1000,C123,S
4,3,"Allen, Mr. William Henry",1,35.0,0,0,373450,8.0500,NaN,S


In [336]:
X = X.drop("Name", axis=1)
X_test = X_test.drop("Name", axis=1)

X = X.drop("Ticket", axis=1)
X_test = X_test.drop("Ticket", axis=1)

X = X.drop("Cabin", axis=1)
X_test = X_test.drop("Cabin", axis=1)

In [337]:
X["Embarked"].value_counts().count()

np.int64(3)

In [338]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [339]:
embarked_mode = X_train["Embarked"].mode()[0]
age_median = X_train["Age"].median()

In [340]:
X_train.fillna({"Embarked": embarked_mode, "Age": age_median}, inplace=True)
X_val.fillna({"Embarked": embarked_mode, "Age": age_median}, inplace=True)
X_test.fillna({"Embarked": embarked_mode, "Age": age_median}, inplace=True)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,1,34.5,0,0,7.8292,Q
1,3,0,47.0,1,0,7.0000,S
2,2,1,62.0,0,0,9.6875,Q
3,3,1,27.0,0,0,8.6625,S
4,3,0,22.0,1,1,12.2875,S
...,...,...,...,...,...,...,...
413,3,1,29.0,0,0,8.0500,S
414,1,0,39.0,0,0,108.9000,C
415,3,1,38.5,0,0,7.2500,S
416,3,1,29.0,0,0,8.0500,S


In [341]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 668 entries, 486 to 821
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    668 non-null    int64  
 1   Sex       668 non-null    int64  
 2   Age       668 non-null    float64
 3   SibSp     668 non-null    int64  
 4   Parch     668 non-null    int64  
 5   Fare      668 non-null    float64
 6   Embarked  668 non-null    str    
dtypes: float64(2), int64(4), str(1)
memory usage: 41.8 KB


In [342]:
X_val.info()

<class 'pandas.DataFrame'>
Index: 223 entries, 157 to 639
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    223 non-null    int64  
 1   Sex       223 non-null    int64  
 2   Age       223 non-null    float64
 3   SibSp     223 non-null    int64  
 4   Parch     223 non-null    int64  
 5   Fare      223 non-null    float64
 6   Embarked  223 non-null    str    
dtypes: float64(2), int64(4), str(1)
memory usage: 13.9 KB


In [343]:
X_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    418 non-null    int64  
 1   Sex       418 non-null    int64  
 2   Age       418 non-null    float64
 3   SibSp     418 non-null    int64  
 4   Parch     418 non-null    int64  
 5   Fare      417 non-null    float64
 6   Embarked  418 non-null    str    
dtypes: float64(2), int64(4), str(1)
memory usage: 23.0 KB


In [344]:
X_test.fillna({"Fare": X_train["Fare"].median()}, inplace=True)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,1,34.5,0,0,7.8292,Q
1,3,0,47.0,1,0,7.0000,S
2,2,1,62.0,0,0,9.6875,Q
3,3,1,27.0,0,0,8.6625,S
4,3,0,22.0,1,1,12.2875,S
...,...,...,...,...,...,...,...
413,3,1,29.0,0,0,8.0500,S
414,1,0,39.0,0,0,108.9000,C
415,3,1,38.5,0,0,7.2500,S
416,3,1,29.0,0,0,8.0500,S


In [345]:
ohe = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
).set_output(transform="pandas")

encoded = ohe.fit_transform(X_train[["Embarked"]])
encoded_val = ohe.transform(X_val[["Embarked"]])
encoded_test = ohe.transform(X_test[["Embarked"]])

X_train = pd.concat([X_train.drop(columns="Embarked"), encoded], axis=1)
X_val = pd.concat([X_val.drop(columns="Embarked"), encoded_val], axis=1)
X_test = pd.concat([X_test.drop(columns="Embarked"), encoded_test], axis=1,)

In [346]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
486,1,0,35.0,1,0,90.000,0.0,0.0,1.0
238,2,1,19.0,0,0,10.500,0.0,0.0,1.0
722,2,1,34.0,0,0,13.000,0.0,0.0,1.0
184,3,0,4.0,0,2,22.025,0.0,0.0,1.0
56,2,0,21.0,0,0,10.500,0.0,0.0,1.0


In [347]:
scaler = StandardScaler()

scaler.fit(X_train)

X_train = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val = pd.DataFrame(scaler.transform(X_val), columns=X_train.columns, index=X_val.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_train.columns, index=X_test.index)

In [348]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
486,-1.579334,-1.361874,0.405333,0.536677,-0.451483,1.240397,-0.498596,-0.29656,0.623517
238,-0.376375,0.734282,-0.829764,-0.464720,-0.451483,-0.436028,-0.498596,-0.29656,0.623517
722,-0.376375,0.734282,0.328140,-0.464720,-0.451483,-0.383310,-0.498596,-0.29656,0.623517
184,0.826584,-1.361874,-1.987668,-0.464720,1.951631,-0.192999,-0.498596,-0.29656,0.623517
56,-0.376375,-1.361874,-0.675377,-0.464720,-0.451483,-0.436028,-0.498596,-0.29656,0.623517


In [349]:
X_test.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
0,0.826584,0.734282,0.366737,-0.464720,-0.451483,-0.492347,-0.498596,3.371998,-1.603806
1,0.826584,-1.361874,1.331656,0.536677,-0.451483,-0.509833,-0.498596,-0.296560,0.623517
2,-0.376375,0.734282,2.489560,-0.464720,-0.451483,-0.453161,-0.498596,3.371998,-1.603806
3,0.826584,0.734282,-0.212215,-0.464720,-0.451483,-0.474776,-0.498596,-0.296560,0.623517
4,0.826584,-1.361874,-0.598183,0.536677,0.750074,-0.398335,-0.498596,-0.296560,0.623517


In [351]:
baseline_pred = pd.Series(
    y_train.mode()[0],
    index=y_val.index,
    name=y_val.name,
)

accuracy_score(y_val, baseline_pred)

0.6143497757847534

In [364]:
model = LogisticRegression()

model.fit(X_train, y_train)
pred = model.predict(X_val)
accuracy_score(y_val, pred)

0.7757847533632287

In [ ]:
test_pred = model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test_ids.to_numpy(),
    "Survived": test_pred.ravel().astype(int),
})

submission.to_csv("../submission_logreg.csv", index=False)

In [ ]:
submission

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [ ]:
submission.to_csv("../submission_logreg.csv", index=False)
# 0.77272

In [384]:
model = LogisticRegression()

X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

model.fit(X_train_val, y_train_val)
test_pred = model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test_ids.to_numpy(),
    "Survived": test_pred.ravel().astype(int),
})

submission.to_csv("../submission_logreg.csv", index=False)